# 04 — Evaluation Results
Load a trained checkpoint and run:
- Generated vs. actual spray charts for 3 example batter archetypes
- Calibration reliability diagram
- KL divergence vs. PA count (sparse-data stress test)

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.inference.sample import load_model, generate
from src.inference.inpaint import inpaint
from src.evaluation.metrics import kl_divergence, zone_accuracy, calibration_score
from src.data.preprocess import normalize_coords
from src.analysis.visualize import plot_comparison, plot_calibration_diagram, plot_kl_vs_pa
from src.evaluation.baselines import HistoricalKDE, SituationalKDE

PROCESSED_DIR = '../data/processed'
CHECKPOINT    = '../checkpoints/best.pt'
RAW_DIR       = '../data/raw'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Load model
model = load_model(CHECKPOINT, PROCESSED_DIR, device=device, inpaint_mode=True)
print('Model loaded.')

In [ ]:
# Load metadata and pick 3 example batters
meta = pd.read_csv(f'{PROCESSED_DIR}/metadata.csv')
test_meta = meta[(meta['split'] == 'test') & (meta['chart_type'] == 'full')]

import json
with open(f'{PROCESSED_DIR}/batter_id_map.json') as f:
    batter_id_map = {int(k): v for k, v in json.load(f).items()}

# Pick batters by PA count: high (established), medium, low (rookie proxy)
test_meta_sorted = test_meta.sort_values('pa_count', ascending=False)
example_rows = [
    test_meta_sorted.iloc[0],    # high-PA pull hitter candidate
    test_meta_sorted.iloc[len(test_meta_sorted)//2],  # median PA spray hitter
    test_meta_sorted.iloc[-1],   # low-PA rookie proxy
]
for r in example_rows:
    print(f"batter={r['batter_id']}  pa={r['pa_count']}  year={r['year']}")

In [ ]:
# Generated vs. actual for each example batter
archetypes = ['Pull hitter', 'Spray hitter', 'Rookie (sparse)']

for label, row in zip(archetypes, example_rows):
    mlbam = int(row['batter_id'])
    bidx  = batter_id_map[mlbam]

    # Generate mean chart (average over 50 samples)
    samples = generate(model, bidx, situation_str='full', num_samples=50,
                       num_inference_steps=200, device=device)
    mean_gen = samples.mean(axis=0)

    # Load actual chart
    actual = np.load(f"{PROCESSED_DIR}/{row['file_path']}")

    fig = plot_comparison(mean_gen, actual, batter_label=f'{label} (MLBAM {mlbam})')
    plt.show()
    print(f'  KL(actual||generated): {kl_divergence(mean_gen, actual):.4f}')

In [ ]:
# Calibration reliability diagram
# Use the first example batter with enough 2023 raw events
row = example_rows[0]
mlbam = int(row['batter_id'])
bidx  = batter_id_map[mlbam]

raw_2023 = pd.read_csv(f'{RAW_DIR}/statcast_2023.csv', low_memory=False)
batter_events = raw_2023[raw_2023['batter'] == mlbam].dropna(subset=['hc_x', 'hc_y'])
x_act, y_act = normalize_coords(batter_events['hc_x'].values, batter_events['hc_y'].values)
actual_coords = np.stack([x_act, y_act], axis=1)

samples = generate(model, bidx, num_samples=200, num_inference_steps=200, device=device)
model_cov = calibration_score(samples, actual_coords, confidence_level=0.80)

# KDE calibration as baseline
hist_kde = HistoricalKDE()
raw_all = pd.concat([pd.read_csv(f'{RAW_DIR}/statcast_{y}.csv', low_memory=False) for y in range(2015,2023)])
hist_kde.fit(raw_all)
kde_density = hist_kde.predict(mlbam, {})
# Calibration for KDE: repeat density as a single 'sample'
kde_cov = calibration_score(kde_density[np.newaxis], actual_coords, confidence_level=0.80)

print(f'Diffusion 80% coverage : {model_cov:.3f}  (target 0.80)')
print(f'HistoricalKDE 80% coverage: {kde_cov:.3f}')

In [ ]:
# KL divergence vs. PA count — sparse-data stress test
full_chart = np.load(f"{PROCESSED_DIR}/{row['file_path']}")
pa_thresholds = [10, 25, 50, 100]

model_kl, kde_kl = {}, {}
hx = batter_events['hc_x'].values
hy = batter_events['hc_y'].values

for k in pa_thresholds:
    if k > len(hx):
        continue
    # Diffusion inpainting
    gen_samples = inpaint(model, bidx, hx, hy, k=k,
                          num_samples=30, num_inference_steps=100, device=device)
    model_kl[k] = kl_divergence(gen_samples.mean(axis=0), full_chart)

    # Situational KDE
    idx = np.random.choice(len(hx), k, replace=False)
    sub_x, sub_y = normalize_coords(hx[idx], hy[idx])
    from src.data.preprocess import coords_to_image
    kde_partial = coords_to_image(sub_x, sub_y)
    kde_kl[k] = kl_divergence(kde_partial, full_chart)

fig = plot_kl_vs_pa({'Diffusion (inpaint)': model_kl, 'KDE partial': kde_kl})
plt.show()